# Building a Character-Level Bigram Language Model from Scratch

In this notebook we'll build the **simplest possible language model**: a bigram model. "Bigram" means the model only ever looks at **one character** to predict the **next character** — no long-range context, no attention, nothing fancy. It's the "hello world" of language modeling, and it's the perfect foundation before moving on to Transformers.

We'll use the **tiny-shakespeare** dataset (a file called `input.txt` containing the complete works of Shakespeare as plain text) and train entirely at the **character level** — meaning our vocabulary is just the set of unique characters (letters, punctuation, spaces, newlines) that appear in the text, rather than words or sub-word tokens.

By the end, you'll have:
1. A character-level tokenizer (encoder/decoder)
2. Train/validation data splits
3. A batching function for sampling training examples
4. A `BigramLanguageModel` built with `nn.Module`
5. A training loop using AdamW
6. A `generate()` method to sample new Shakespeare-flavored text from the model

Let's go step by step.

## Step 1: Building the Vocabulary

Before a neural network can process text, we need to convert that text into numbers. The very first step is figuring out **what symbols exist in our dataset** — this set of symbols is called the **vocabulary**.

Since we're building a *character-level* model (as opposed to a word-level or subword/BPE-level model like GPT uses), our vocabulary will simply be every unique character that appears anywhere in `input.txt`. That includes:
- Lowercase and uppercase letters (`a`–`z`, `A`–`Z`)
- Punctuation (`.`, `,`, `!`, `?`, `:`, `;`, etc.)
- Whitespace characters (spaces, newlines `\n`)
- Any other symbol present in the raw text

**Why sort the characters?** Sorting gives us a *deterministic* vocabulary. If we didn't sort, the order of `set()` in Python could vary between runs (technically `set` order is insertion-dependent and not guaranteed), which would mean every time we re-run this notebook, character `'a'` might map to a different integer ID. Sorting fixes the order once and for all, so our token IDs are reproducible.

The size of this vocabulary (let's call it `vocab_size`) is an important number — it will define:
- The output dimension of our model (since the model must predict a probability distribution over **all possible next characters**)
- The size of our embedding table

Let's read the file and build this vocabulary.

In [1]:
# Read the entire dataset into memory as a single string
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text):,}")

# The vocabulary is the set of unique characters that occur in this text,
# sorted so the ordering (and therefore the integer ID of each character) is deterministic
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print("Vocabulary:", ''.join(chars))

Length of dataset in characters: 1,115,394
Vocabulary size: 65
Vocabulary: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


## Step 2: Building the Tokenizer

Neural networks operate on numbers (tensors), not raw text. A **tokenizer** is the bridge between human-readable text and the integer representations a model can actually compute with.

Since we've already established our vocabulary (the sorted list of unique characters), building a character-level tokenizer is straightforward. We need two lookup tables:

- **`stoi`** ("string to integer"): maps each character to its unique integer ID, based on its position in the sorted vocabulary list (e.g. `'a'` → `13`).
- **`itos`** ("integer to string"): the inverse mapping — given an integer ID, return the original character (e.g. `13` → `'a'`).

With these two dictionaries, we define two functions:
- **`encode(s)`**: takes a Python string `s` and returns a list of integers (one per character), using `stoi`.
- **`decode(l)`**: takes a list of integers `l` and returns the reconstructed string, using `itos`.

This is admittedly a *very* simple tokenizer compared to what production LLMs use (like Byte-Pair Encoding / BPE, which GPT models use to tokenize sub-word chunks). Character-level tokenization has a tiny vocabulary (usually under 100 symbols) but produces longer sequences, since each character is its own token. It's a great starting point because it keeps the rest of the pipeline simple and easy to reason about.

Let's build it and sanity-check it with a round-trip test: encode a string, decode it back, and confirm we get the original string.

In [2]:
# Create the mapping from characters to integers and back
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }

def encode(s):
    # Take a string, return a list of integers (token IDs)
    return [stoi[c] for c in s]

def decode(l):
    # Take a list of integers (token IDs), return the decoded string
    return ''.join([itos[i] for i in l])

# Quick test: encode then decode a small string and verify we get it back exactly
test_string = "Hello there!"
encoded = encode(test_string)
decoded = decode(encoded)

print(f"Original string: {test_string!r}")
print(f"Encoded (token IDs): {encoded}")
print(f"Decoded string: {decoded!r}")
print(f"Round-trip successful: {decoded == test_string}")

Original string: 'Hello there!'
Encoded (token IDs): [20, 43, 50, 50, 53, 1, 58, 46, 43, 56, 43, 2]
Decoded string: 'Hello there!'
Round-trip successful: True


## Step 3: Encoding the Entire Dataset

Now that we have a working tokenizer, we can encode the **entire** Shakespeare text into a single, long sequence of integers. We'll store this as a **1D PyTorch tensor**.

Why a tensor and not just a Python list? Because PyTorch tensors are the native data structure for all the math we're about to do — efficient slicing, GPU acceleration, batched operations, and integration with `nn.Module` layers all expect tensors.

We use `dtype=torch.long` (i.e. 64-bit integer) because:
- Token IDs are inherently discrete categorical indices, not continuous floating point values.
- PyTorch's embedding lookup (`nn.Embedding`) and the cross-entropy loss function (`F.cross_entropy`) both specifically expect integer *index* tensors of type `long` for this purpose.

After encoding, this tensor will be **one giant 1D sequence** — there's no notion of "batches" or "sentences" yet, just one long stream of character IDs representing the whole corpus, in order.

In [3]:
import torch

# Encode the entire text dataset into a tensor of integers
data = torch.tensor(encode(text), dtype=torch.long)

print(f"Shape of encoded data tensor: {data.shape}")
print(f"Dtype: {data.dtype}")

# Peek at the first 100 tokens, and what they decode back to
print("\nFirst 100 token IDs:")
print(data[:100])
print("\nWhich decode back to:")
print(repr(decode(data[:100].tolist())))

Shape of encoded data tensor: torch.Size([1115394])
Dtype: torch.int64

First 100 token IDs:
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

Which decode back to:
'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'


## Step 4: Train / Validation Split

Before training, we need to hold out a portion of our data to monitor **generalization** — that is, to check whether the model is actually learning the underlying patterns of the English language (and Shakespeare's style), rather than just memorizing the exact training sequences.

We'll use a standard **90/10 split**:
- **90%** of the data → training set (`train_data`): the model directly learns from this.
- **10%** of the data → validation set (`val_data`): the model never trains on this; we only use it to *evaluate* the loss periodically.

If the training loss keeps dropping but the validation loss stalls or rises, that's a classic sign of **overfitting** — the model is starting to memorize quirks of the training data rather than learning generalizable structure.

Note that because our data is one long *contiguous* sequence of characters (the text in its original reading order), we do the split by simply slicing the tensor at the 90% mark — everything before that index is training data, everything after is validation data. We are **not** randomly shuffling individual characters; that would destroy the sequential structure the model needs to learn from.

In [4]:
# Split the data into a training set (90%) and a validation set (10%)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Total tokens: {len(data):,}")
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens:   {len(val_data):,}")

Total tokens: 1,115,394
Train tokens: 1,003,854
Val tokens:   111,540


## Step 5: Batching the Data with `get_batch`

We can't feed the *entire* dataset into the model in one shot — it's far too large, and training works much better with small, randomly-sampled chunks anyway (this is the basis of **stochastic gradient descent**, where we estimate the true gradient using small random batches rather than the full dataset every step).

We need two hyperparameters:
- **`block_size` (T)**: the length of each contiguous chunk of text we feed into the model. Think of this as the model's "context window" — how many previous characters it gets to see before predicting the next one. (Even though our *bigram* model will technically only use the single most recent character, we still organize the data into chunks of length `T` so that this same data-loading code will work unmodified when we upgrade to fancier models later, like a Transformer, that *do* use the full context window.)
- **`batch_size` (B)**: how many independent chunks we process simultaneously in one training step, stacked together for efficient parallel computation on the GPU/CPU.

**How `get_batch` works:**
1. Randomly choose `B` starting indices into the dataset (using `torch.randint`).
2. For each starting index `i`, take the chunk `data[i : i+T]` — this becomes one row of our input `x`.
3. For the corresponding target `y`, take the chunk shifted **one position to the right**: `data[i+1 : i+T+1]`. This means at every position `t` in the sequence, `y[t]` is exactly the character that comes right after `x[t]` — i.e., the "correct answer" the model should predict.

Concretely, this is why we need chunks of length `T+1` "under the hood": we slice out `T+1` consecutive characters, then use the first `T` as `x` and the last `T` (shifted by one) as `y`. Both `x` and `y` end up with shape `(B, T)`.

This setup means a single chunk of length `T` actually gives the model `T` separate training examples at once: predict character 2 from character 1, predict character 3 from characters 1-2, predict character 4 from characters 1-3, and so on — all packed efficiently into one tensor operation.

In [5]:
def get_batch(split):
    # Sample a random batch of data of inputs x and targets y.
    # split: either 'train' or 'val'
    # Returns:
    #   x: (batch_size, block_size) tensor of input token IDs
    #   y: (batch_size, block_size) tensor of target token IDs (x shifted by 1)
    data_split = train_data if split == 'train' else val_data

    # Randomly choose `batch_size` starting indices.
    # We subtract block_size so that i + block_size never runs off the end of the data.
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # For each starting index i, grab the chunk of length block_size (this is x),
    # and the same chunk shifted one to the right (this is y, the "next character" targets)
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x, y

# Let's try it out with small values first, just to inspect the shapes and contents
batch_size = 4
block_size = 8

xb, yb = get_batch('train')
print("inputs (x):")
print(xb)
print("shape:", xb.shape)
print("\ntargets (y):")
print(yb)
print("shape:", yb.shape)

# Sanity check: for the first sequence in the batch, show how x predicts y position by position
print("\n--- Example: how x[0] maps to y[0] ---")
for t in range(block_size):
    context = xb[0, :t+1]
    target = yb[0, t]
    print(f"when input is {context.tolist()} the target is {target.item()} ({itos[target.item()]!r})")

inputs (x):
tensor([[43, 57, 43,  1, 53, 59, 58, 56],
        [51, 53, 57, 58,  1, 56, 53, 63],
        [58, 50, 43, 51, 39, 52, 10,  0],
        [28, 53, 57, 58, 10,  0,  0, 27]])
shape: torch.Size([4, 8])

targets (y):
tensor([[57, 43,  1, 53, 59, 58, 56, 39],
        [53, 57, 58,  1, 56, 53, 63, 39],
        [50, 43, 51, 39, 52, 10,  0, 20],
        [53, 57, 58, 10,  0,  0, 27, 36]])
shape: torch.Size([4, 8])

--- Example: how x[0] maps to y[0] ---
when input is [43] the target is 57 ('s')
when input is [43, 57] the target is 43 ('e')
when input is [43, 57, 43] the target is 1 (' ')
when input is [43, 57, 43, 1] the target is 53 ('o')
when input is [43, 57, 43, 1, 53] the target is 59 ('u')
when input is [43, 57, 43, 1, 53, 59] the target is 58 ('t')
when input is [43, 57, 43, 1, 53, 59, 58] the target is 56 ('r')
when input is [43, 57, 43, 1, 53, 59, 58, 56] the target is 39 ('a')


## Step 6: The Bigram Language Model

Now for the core model itself. A **bigram model** predicts the next character using *only* the current character — it has no memory of anything earlier in the sequence. Despite this extreme simplicity, it's a great way to build intuition for the machinery (embeddings, logits, cross-entropy loss, sampling) that every more sophisticated language model — up to and including GPT — also relies on.

**The key trick:** we implement this using a single `nn.Embedding(vocab_size, vocab_size)` layer.

Normally, an embedding layer maps a discrete token ID to a dense *vector* of some chosen size (e.g., mapping vocab_size → 64-dimensional vectors). But here we deliberately choose the embedding dimension to **also be `vocab_size`**. This means: for every possible input character, the embedding table directly stores a full row of `vocab_size` numbers — and we interpret that row *directly as the logits* (unnormalized scores) for what the next character should be.

In other words: row `i` of the embedding table *is* the model's lookup table answering "given that the current character is token `i`, what's my raw score for every possible next character?" There's no hidden layer, no nonlinearity — it's literally a lookup table of next-character scores, learned via gradient descent.

**The forward pass:**
1. Input `idx` has shape `(B, T)` — a batch of `B` sequences, each of length `T` token IDs.
2. We pass `idx` through the embedding table, producing `logits` of shape `(B, T, C)`, where `C = vocab_size`. For every single position in every sequence, we now have a vector of `C` raw scores — one score per possible next character.
3. **If targets are provided**, we compute the **cross-entropy loss** between the predicted logits and the actual next-character targets `y`. Cross-entropy loss measures how far off our predicted probability distribution is from the "correct" answer (it's mathematically equivalent to the negative log-likelihood of the true next token under our predicted distribution). Lower is better — a perfect model would assign all probability mass to the correct character.
4. **If no targets are given** (i.e., we're just generating, not training), we skip the loss computation and return `None` for it.

One implementation detail: PyTorch's `F.cross_entropy` expects its input as `(N, C)` (N total predictions, C classes) and its target as `(N,)` — but our `logits` tensor is shaped `(B, T, C)` and our targets are `(B, T)`. So inside `forward`, we `.view()` (reshape) both tensors to flatten the batch and time dimensions together before computing the loss.

In [6]:
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)  # for reproducibility

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table.
        # Shape of the embedding table's weight: (vocab_size, vocab_size)
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensors of integers
        logits = self.token_embedding_table(idx)  # shape: (B, T, C) where C = vocab_size

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # Flatten batch and time dimensions so cross_entropy sees (N, C) and (N,)
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

# Quick sanity check: run one forward pass on the batch we created above
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print("logits shape:", logits.shape)
print("loss:", loss.item())

# Sanity check on the loss: with vocab_size classes and random initial weights,
# we'd expect the initial loss to be roughly -ln(1/vocab_size) if predictions start near-uniform
import math
expected_initial_loss = -math.log(1.0 / vocab_size)
print(f"Expected loss if predictions were uniform: {expected_initial_loss:.4f}")

logits shape: torch.Size([4, 8, 65])
loss: 5.2601189613342285
Expected loss if predictions were uniform: 4.1744


## Step 7: Training the Model

Now we train the model so that the embedding table's rows stop being random noise and start actually encoding real character-transition statistics learned from Shakespeare's text (e.g., learning that `'q'` is almost always followed by `'u'`, that a newline often follows a period, etc.).

**The optimizer — AdamW:** We use `torch.optim.AdamW`, a very popular and robust variant of the Adam optimizer (it improves on Adam's handling of weight decay / L2 regularization). It adaptively adjusts the learning rate for each parameter based on the history of its gradients, which generally makes training faster and more stable than plain stochastic gradient descent. We set the learning rate to `1e-2` (0.01) — a relatively high learning rate that works well for this tiny model.

**The training loop, step by step:**
1. Sample a fresh random batch `(xb, yb)` using `get_batch('train')`.
2. Run the forward pass: `logits, loss = model(xb, yb)`.
3. **Zero out old gradients** with `optimizer.zero_grad(set_to_none=True)`. PyTorch accumulates gradients by default, so we must clear them before each new backward pass, or gradients from previous steps would incorrectly add up.
4. **Backpropagate**: `loss.backward()` computes the gradient of the loss with respect to every trainable parameter (here, just the embedding table's weights).
5. **Update parameters**: `optimizer.step()` nudges every parameter slightly in the direction that reduces the loss, scaled by the learning rate and Adam's adaptive per-parameter scaling.
6. Periodically, we also check the loss on the **validation set** (without updating weights) to monitor generalization.

We'll train for **3000 steps**, with `batch_size = 32` and `block_size = 8`, and print the train/validation loss every few hundred steps so we can watch it improve.

In [7]:
# Hyperparameters for training
batch_size = 32
block_size = 8
learning_rate = 1e-2
max_steps = 3000
eval_interval = 300  # how often we print the loss

# Re-create the model fresh (so this cell can be re-run cleanly) and set up the optimizer
model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()  # disable gradient tracking, since this is just evaluation, not training
def estimate_loss(eval_iters=50):
    # Average the loss over several batches for both train and val splits, for a less noisy estimate.
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

for step in range(max_steps):
    # Periodically report the train/val loss
    if step % eval_interval == 0 or step == max_steps - 1:
        losses = estimate_loss()
        print(f"step {step:>5}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of training data
    xb, yb = get_batch('train')

    # Evaluate the loss and backpropagate
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\nTraining complete!")

step     0: train loss 4.6180, val loss 4.6240
step   300: train loss 2.7808, val loss 2.8092
step   600: train loss 2.5265, val loss 2.5657
step   900: train loss 2.4945, val loss 2.5210
step  1200: train loss 2.4934, val loss 2.5121
step  1500: train loss 2.4777, val loss 2.4975
step  1800: train loss 2.4710, val loss 2.4903
step  2100: train loss 2.4701, val loss 2.4938
step  2400: train loss 2.4545, val loss 2.4866
step  2700: train loss 2.4646, val loss 2.4866
step  2999: train loss 2.4485, val loss 2.4813

Training complete!


## Step 8: Generating Text from the Model

The final piece is letting the model **generate** new text by repeatedly predicting and sampling the next character, one at a time. This is exactly how all autoregressive language models (including GPT) generate text at inference time — just predict one token, append it, and feed the longer sequence back in for the next prediction.

**The `generate` method, step by step:**
1. We start with some initial sequence `idx` of shape `(B, T)` — this could even be just a single token (e.g., a "start of sequence" placeholder).
2. We loop `max_new_tokens` times. In each iteration:
   - Run the forward pass to get `logits` of shape `(B, T, C)`.
   - We only care about the prediction for the **very last position** in the sequence (since that's the "next character" prediction): `logits[:, -1, :]`, giving us shape `(B, C)`.
   - Apply **softmax** to convert these raw logits into a proper probability distribution over the vocabulary (`F.softmax`). Softmax exponentiates each logit and normalizes so all probabilities sum to 1 — guaranteeing a valid probability distribution.
   - Use **`torch.multinomial`** to randomly sample one token ID from this probability distribution. This is "categorical sampling" — rather than greedily always picking the single highest-probability character (which tends to produce repetitive, deterministic text), we sample proportionally to the predicted probabilities, which introduces natural variety while still favoring likely characters.
   - **Append** this newly sampled token to the running sequence `idx` using `torch.cat`.
3. After `max_new_tokens` iterations, `idx` has grown from its original length to `original_length + max_new_tokens`, and we return the whole thing.

Note: because our bigram model only ever looks at the single most recent token to make its prediction, feeding it a longer and longer sequence doesn't actually change its behavior at each step (it always just looks at `logits[:, -1, :]`, which is purely a function of the *last* token in `idx`). This will matter much more once we move to context-aware models like Transformers, where the *entire* sequence so far influences the next prediction. For now, this generate loop is written in a general way that will keep working unchanged when we eventually upgrade the model.

Let's add this method to our model class. Since we're redefining the `BigramLanguageModel` class to include this new method, we'll also retrain a fresh instance using the exact same settings as Step 7 — that way we have a properly trained model (`model`) that also has `generate` available on it.

In [8]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx: (B, T) tensor of token IDs, the current context
        # Generates max_new_tokens new tokens, appended one at a time, and returns
        # the full sequence of shape (B, T + max_new_tokens)
        for _ in range(max_new_tokens):
            # Get predictions for the current sequence
            logits, _ = self(idx)              # (B, T, C)

            # Focus only on the last time step -- that's the "next token" prediction
            logits = logits[:, -1, :]           # (B, C)

            # Convert logits to probabilities
            probs = F.softmax(logits, dim=-1)   # (B, C)

            # Sample the next token from this probability distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)

            # Append the sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)

        return idx


# Retrain a fresh instance of this (now generate-capable) class, using the same
# hyperparameters and training loop as Step 7, so `model` below is both trained
# AND has the generate() method available.
model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_steps):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

final_losses = estimate_loss()
print(f"Final train loss: {final_losses['train']:.4f}, final val loss: {final_losses['val']:.4f}")

Final train loss: 2.4695, final val loss: 2.4788


### Comparing Untrained vs. Trained Generation

Let's instantiate a **fresh, untrained** model and generate from it first — since its embedding table is just random numbers, we expect complete gibberish. Then we'll generate from our **trained** model and see it has actually picked up some character-level statistics of English/Shakespearean text (don't expect coherent words — it's "just" a bigram model — but you should see things like more plausible letter combinations, spaces appearing in sensible places, and capital letters starting after newlines).

We start generation from a single token with ID `0` as a simple seed/starting context (shape `(1, 1)`), then ask for several hundred new characters.

In [9]:
# A starting context of a single token (ID 0), as a batch of size 1
context = torch.zeros((1, 1), dtype=torch.long)

print("="*60)
print("GENERATION FROM AN UNTRAINED MODEL (random weights)")
print("="*60)
untrained_model = BigramLanguageModel(vocab_size)
generated_ids = untrained_model.generate(context, max_new_tokens=300)[0].tolist()
print(decode(generated_ids))

print("\n" + "="*60)
print("GENERATION FROM OUR TRAINED MODEL")
print("="*60)
trained_generated_ids = model.generate(context, max_new_tokens=300)[0].tolist()
print(decode(trained_generated_ids))

GENERATION FROM AN UNTRAINED MODEL (random weights)

$3mUUxeBNVJmF'HGrqpanQw,uxAaDcNmVs.YLtjycybTx,pRJcybJq-C;:lw,VQ v,I$FHxan,jeTjeq
iGmK$xLF:siI;HHMPIr!?pkeKaYiuwtTnXWcN w,LO-!KPQH$ZtDk:E,lT-ATnQ'Y-.-BBUcCo3fKy?-C,?ibSMmw,tDy-dajUuXhH!VxLoNjz$z?,SHJM&:lyjHjN
rhuDqJaUV-B.LcejPzcYzgSN J&mxJjcppBeJRFmElseVOl:lDVda'!OJiyqYGDSLwDmw$stW,UAD33C,q
lmLZDnz-L

GENERATION FROM OUR TRAINED MODEL

OUSe h h'th s w t.

PENesig d weanondw m.
JOLerisur he
ARIDUSe memarad--sethathis h hianghe llk t at g was jede d
TI w tr fipre wiomeshicim.
Burinere gowe
Bave ft se trthit I str-r mur,
ThIfavee
DUpth herey f aked th.
o'd, m hese a d d thelllinugo,
Ange n s faur t or d t, sipreve.

O:


C. d O: 'd, 


## Wrapping Up

You've now built a complete (if intentionally simple) character-level language model pipeline from scratch:

- **Tokenization**: turning raw text into integer IDs and back
- **Data pipeline**: train/val splits and random batch sampling
- **Model**: a single embedding table acting as a next-character lookup table
- **Training**: AdamW optimization driving down cross-entropy loss
- **Generation**: autoregressive sampling with softmax + multinomial sampling

The output text is still far from coherent English — that's expected! A bigram model has a *severe* limitation: it only ever considers the single most recent character. It can never know that it's in the middle of a word, what was said three words ago, or anything about grammar or meaning beyond raw single-character transition frequencies.

This is, however, exactly the right place to start. Every concept here — embeddings, logits, cross-entropy loss, the training loop, autoregressive generation — carries over directly into far more powerful models. The natural next step is to give the model **access to more context** (more than just the last character) and a way to **weigh different parts of that context differently** — which is precisely what self-attention and the Transformer architecture (the "T" in GPT) provide. If you continue this learning path, that's the next concept to tackle.